# M8 on Kaggle: full-pool FedAvg and FedProx (E2), with the learning rate chosen on this pool

**Why the sweep:** lr 0.1 was chosen on the dev pool, where a round has 11 to 18 times fewer steps, and it made FedAvg unstable on the full pool (0.83 and 0.61 on window average, seeds 42 and 43). This notebook first chooses the rate on the full pool's validation rows, then runs every pass with it, for seeds 42 and 43. One click does all of it.

**Attach as input:** the same private Kaggle dataset as `m3_kaggle.ipynb` (the five files of the M1 `processed/` folder).

**Settings:** Internet *On*. Accelerator: *None*. The MLP trains on CPU, as the README's M8 plan says.
**Run:** *Save Version → Save & Run All (Commit)* so it keeps running with the browser closed. An interactive session can stop when it is left alone.
Time is an estimate: 8 sweep passes and 12 final passes at about 9 minutes each, and lower rates may take more rounds, so allow 3 to 6 hours.

Each pass runs in its own process, and the zip is rewritten after each stage, so a failure does not lose what finished. Every pass prints an exit code: 137 means the process was killed (usually out of memory). `--report` then writes the report from whatever finished, and says which passes are missing.

**Output:** `/kaggle/working/m8_full_outputs.zip` contains `reports/m8/full/` (`LR_SWEEP.csv`, seed 42, `seed43/`) and the run folders. Download it and bring it back, also after a failure.


In [ ]:
SEEDS = [42, 43]
import os, subprocess, sys
REPO = "/kaggle/working/IDS-code"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Indrajit147/IDS-code.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
# --no-deps: keep Kaggle's preinstalled stack intact; versions are recorded per run.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO, "--no-deps"], check=True)
os.chdir(REPO)
os.environ["IDS_REPO_ROOT"] = REPO
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Rebuild the M2 partition for the full pool (deterministic, ~1 min)
!python scripts/m2_build_partitions.py --env kaggle --pools full
# M2 redraws these figures, and their bytes differ on Kaggle. Nothing reads them. A changed report or JSON stays flagged.
!git checkout -- 'reports/m2/full/*.png'


In [ ]:
import glob, os, shutil
import pandas as pd
REP = "reports/m8/full"
def pack():
    stage = "/kaggle/working/m8_stage"
    shutil.rmtree(stage, ignore_errors=True)
    shutil.copytree(REP, f"{stage}/{REP}", ignore=shutil.ignore_patterns("lr0.1"))
    for r in sorted(glob.glob("/kaggle/working/results/runs/*_m8_fedavg_full")):
        shutil.copytree(r, f"{stage}/results/runs/{os.path.basename(r)}")
    shutil.make_archive("/kaggle/working/m8_full_outputs", "zip", stage)
    shutil.rmtree(stage)
    print("zip written:", sorted(os.listdir(REP)))


In [ ]:
# Choose the rate on validation rows only. lr 0.1 is repeated, which also checks that the earlier run reproduces.
!python scripts/m8_lr_sweep.py --env kaggle --pool full --lrs 0.003 0.01 0.03 0.1 --seeds 42 43; echo "exit code: $?"
sweep = pd.read_csv(f"{REP}/LR_SWEEP.csv")
LR = float(sweep[sweep.chosen].lr.iloc[0])
print("chosen lr:", LR)
pack()


In [ ]:
# All passes at that rate, then the report. Seed 42 is the main report, as in M7.
for SEED in SEEDS:
    TAG_ARG = "" if SEED == 42 else f"--tag seed{SEED}"
    for name in ["fedavg", "fedavg_no_rehearsal", "fedprox_mu0.001", "fedprox_mu0.01", "fedprox_mu0.1", "fedprox_mu1"]:
        print("=====", SEED, name, flush=True)
        !python -X faulthandler scripts/m8_fedavg.py --env kaggle --pool full --set resources.n_jobs=4 seed={SEED} fedavg.lr={LR} {TAG_ARG} --only {name} --resume; echo "exit code: $?"
        pack()
    !python -X faulthandler scripts/m8_fedavg.py --env kaggle --pool full --set resources.n_jobs=4 seed={SEED} fedavg.lr={LR} {TAG_ARG} --report; echo "exit code: $?"
    pack()
print(open(f"{REP}/FEDAVG.md", encoding="utf-8").read()[:6000])
